In [1]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pandas as pd
import math
from copy import deepcopy

In [2]:
DEBTS = [
    {
        "id": "cc_sbi",
        "name": "SBI Credit Card",
        "category": "revolving",
        "type": "credit_card",
        "balance": 50000,
        "apr": 42.0,
        "min_payment": 2500,
        "emi": None,
        "tenure_months": None,
        "moratorium": None
    },
    {
        "id": "loan_home_hdfc",
        "name": "HDFC Home Loan",
        "category": "installment",
        "type": "loan",
        "balance": 2500000,
        "apr": 8.5,
        "min_payment": None,
        "emi": 21797,
        "tenure_months": 240,
        "moratorium": False
    },
    {
        "id": "loan_car_axis",
        "name": "Axis Car Loan",
        "category": "installment",
        "type": "loan",
        "balance": 600000,
        "apr": 9.5,
        "min_payment": None,
        "emi": 12618,
        "tenure_months": 60,
        "moratorium": False
    },
    {
        "id": "loan_edu",
        "name": "Education Loan",
        "category": "installment",
        "type": "loan",
        "balance": 400000,
        "apr": 10.5,
        "min_payment": None,
        "emi": 6890,
        "tenure_months": 84,
        "moratorium": False
    }
]


In [3]:
MONTHLY_BUDGET = 60000
EXTRA_MONTHLY = 10000

LUMP_SUM_PAYMENTS = [
    {
        "id": "bonus_diwali",
        "month": 1,
        "amount": 5000,
        "apply_to": "target"
    },
    {
        "id": "bonus_year_end",
        "month": 2,
        "amount": 20000,
        "apply_to": "target"
    }
]


In [4]:
import math
from copy import deepcopy
from datetime import datetime
from dateutil.relativedelta import relativedelta

def calculate_emi(principal, apr, months):
    if principal <= 0 or months <= 0:
        return 0
    r = apr / 1200
    if r == 0:
        return principal / months
    return principal * r * ((1 + r) ** months) / ((1 + r) ** months - 1)

def select_target(debts, strategy):
    active = [d for d in debts if d["balance"] > 0]
    if not active:
        return None
    if strategy == "avalanche":
        return max(active, key=lambda d: d["apr"])
    if strategy == "snowball":
        return min(active, key=lambda d: d["balance"])
    if strategy == "hybrid":
        high_apr = [d for d in active if d["apr"] > 15]
        return max(high_apr, key=lambda d: d["apr"]) if high_apr else min(active, key=lambda d: d["balance"])
    return active[0]

def simulate_payoff(debts, strategy, extra_monthly, lump_sums):
    debts = deepcopy(debts)
    month = 0
    total_interest = 0
    timeline = []

    while any(d["balance"] > 0 for d in debts):
        month += 1
        if month > 600:
            break

        month_interest = 0
        month_principal = 0

        for d in debts:
            if d["balance"] <= 0 or d.get("moratorium"):
                continue
            interest = d["balance"] * (d["apr"] / 1200)
            d["balance"] += interest
            month_interest += interest

        for d in debts:
            if d["balance"] <= 0:
                continue
            min_pay = d.get("emi", 0) if d["type"] == "loan" else max(d.get("min_payment", 0), d["balance"] * 0.05)
            payment = min(min_pay, d["balance"])
            d["balance"] -= payment
            month_principal += payment

        lump_sum = sum(ls["amount"] for ls in lump_sums if ls["month"] == month)
        extra_available = extra_monthly + lump_sum

        while extra_available > 0:
            target = select_target(debts, strategy)
            if not target:
                break
            payment = min(extra_available, target["balance"])
            target["balance"] -= payment
            month_principal += payment
            extra_available -= payment

        total_interest += month_interest

        timeline.append({
            "month": month,
            "total_balance": round(sum(d["balance"] for d in debts), 2),
            "interest_paid": round(month_interest, 2),
            "principal_paid": round(month_principal, 2)
        })

    return month, round(total_interest, 2), timeline, debts

def format_duration(months):
    y, m = divmod(months, 12)
    if y == 0:
        return f"{m} months"
    if m == 0:
        return f"{y} years"
    return f"{y} years, {m} months"

def get_debt_free_date(months_from_now):
    return (datetime.now() + relativedelta(months=months_from_now)).strftime("%B %Y")


In [5]:
import pandas as pd

print("="*80)
print("DEBT PAYOFF STRATEGY COMPARISON".center(80))
print("="*80)

print(f"\nYour Situation:")
print(f"  Total Debt:        ₹{sum(d['balance'] for d in DEBTS):,.0f}")
print(f"  Monthly Budget:    ₹{MONTHLY_BUDGET:,.0f}")
print(f"  Extra Monthly:     ₹{EXTRA_MONTHLY:,.0f}")
print(f"  Planned Windfalls: ₹{sum(ls['amount'] for ls in LUMP_SUM_PAYMENTS):,.0f}")

strategies = {
    "Avalanche": "avalanche",
    "Snowball": "snowball",
    "Hybrid": "hybrid"
}

results = {}
for name, strategy in strategies.items():
    months, interest, timeline, final_debts = simulate_payoff(
        DEBTS, strategy, EXTRA_MONTHLY, LUMP_SUM_PAYMENTS
    )
    results[name] = {"months": months, "interest": interest, "timeline": timeline}

best_strategy = min(results.items(), key=lambda x: x[1]["interest"])
baseline = results["Avalanche"]

comparison_data = []
for name, result in results.items():
    savings_vs_baseline = baseline["interest"] - result["interest"]
    time_diff = baseline["months"] - result["months"]
    comparison_data.append({
        "Strategy": name,
        "Payoff Time": format_duration(result["months"]),
        "Debt-Free Date": get_debt_free_date(result["months"]),
        "Total Interest": f"₹{result['interest']:,.0f}",
        "Savings vs Avalanche": "Baseline" if name == "Avalanche" else f"₹{savings_vs_baseline:+,.0f}",
        "Time Saved": "-" if name == "Avalanche" else f"{time_diff:+d} months"
    })

df_comparison = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("STRATEGY COMPARISON".center(80))
print("="*80)
print(df_comparison.to_string(index=False))

print("\n" + "="*80)
print(f"RECOMMENDED STRATEGY: {best_strategy[0]}")
print("="*80)
print(f"  Debt-Free Date:    {get_debt_free_date(best_strategy[1]['months'])}")
print(f"  Total Payoff Time: {format_duration(best_strategy[1]['months'])}")
print(f"  Total Interest:    ₹{best_strategy[1]['interest']:,.0f}")
print(f"  Monthly Payment:   ₹{MONTHLY_BUDGET:,.0f} (+₹{EXTRA_MONTHLY:,.0f} extra)")

print("\n" + "="*80)
print("ATTACK ORDER (Avalanche Method)".center(80))
print("="*80)
sorted_debts = sorted(DEBTS, key=lambda d: d["apr"], reverse=True)
for i, debt in enumerate(sorted_debts, 1):
    debt_type = "Credit Card" if debt["type"] == "credit_card" else "Loan"
    print(f"{i}. {debt_type:<12} | {debt['name']:<25} | APR: {debt['apr']:5.1f}% | Balance: ₹{debt['balance']:>10,.0f}")

print("\n" + "="*80)
print("OPTIMIZATION TIPS".center(80))
print("="*80)

test_extras = [5000, 10000, 15000, 20000]
print("\nImpact of Extra Monthly Payments:")
for extra in test_extras:
    months, interest, _, _ = simulate_payoff(DEBTS, "avalanche", extra, LUMP_SUM_PAYMENTS)
    savings = baseline["interest"] - interest
    time_saved = baseline["months"] - months
    print(f"  +₹{extra:>6,}/month → Save ₹{savings:>8,.0f} | Free {time_saved:>2d} months earlier")

print("\n" + "="*80)
print("DEBT BREAKDOWN".center(80))
print("="*80)
total_debt = sum(d["balance"] for d in DEBTS)
cc_debt = sum(d["balance"] for d in DEBTS if d["type"] == "credit_card")
loan_debt = sum(d["balance"] for d in DEBTS if d["type"] == "loan")

print(f"Credit Cards: ₹{cc_debt:>12,.0f} ({cc_debt/total_debt*100:.1f}%)")
print(f"Loans:        ₹{loan_debt:>12,.0f} ({loan_debt/total_debt*100:.1f}%)")
print(f"Total Debt:   ₹{total_debt:>12,.0f}")

print("\n" + "="*80)
print("TO CUSTOMIZE".center(80))
print("="*80)
print("1. Edit DEBTS list - add/remove/update your debts")
print("2. Set MONTHLY_BUDGET - total you can afford")
print("3. Set EXTRA_MONTHLY - amount beyond minimums")
print("4. Add LUMP_SUM_PAYMENTS - bonuses, tax refunds, etc.")
print("5. Run and compare strategies!")
print("="*80)


                        DEBT PAYOFF STRATEGY COMPARISON                         

Your Situation:
  Total Debt:        ₹3,550,000
  Monthly Budget:    ₹60,000
  Extra Monthly:     ₹10,000
  Planned Windfalls: ₹25,000

                              STRATEGY COMPARISON                               
 Strategy        Payoff Time Debt-Free Date Total Interest Savings vs Avalanche Time Saved
Avalanche 12 years, 2 months     March 2038     ₹1,870,895             Baseline          -
 Snowball 12 years, 2 months     March 2038     ₹1,870,895                  ₹+0  +0 months
   Hybrid 12 years, 2 months     March 2038     ₹1,870,895                  ₹+0  +0 months

RECOMMENDED STRATEGY: Avalanche
  Debt-Free Date:    March 2038
  Total Payoff Time: 12 years, 2 months
  Total Interest:    ₹1,870,895
  Monthly Payment:   ₹60,000 (+₹10,000 extra)

                        ATTACK ORDER (Avalanche Method)                         
1. Credit Card  | SBI Credit Card           | APR:  42.0% | Balance: ₹  